In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
import os
import matplotlib.pyplot as plt
import cortex
import seaborn as sns
from os.path import join
from collections import defaultdict
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import dvu
from neuro.flatmaps_helper import load_flatmaps
from neuro.features.questions.gpt4 import QS_35_STABLE
import sys
from sklearn.metrics import pairwise_distances
import warnings
sys.path.append('../notebooks')
from tqdm import tqdm
from neuro import config
from neuro import analyze_helper
import neuro.viz
from neuro.features.qa_questions import get_questions, get_merged_questions_v3_boostexamples
# flatmaps_per_question = __import__('06_flatmaps_per_question')
# import viz
# import gct
from neuro.flatmaps_helper import load_flatmaps
from statsmodels.stats.multitest import multipletests

Note, this notebook requires first running `03_export_qa_flatmaps.ipynb` into `df_qa_dict.pkl` files for each subject.

### load gemv average flatmaps

In [ ]:
subject = 'S02'
gemv_flatmaps_dict_S02, gemv_flatmaps_dict_S03 = load_flatmaps(
    normalize_flatmaps=False, load_timecourse=False)
if subject == 'S02':
    gemv_flatmaps_dict = gemv_flatmaps_dict_S02
elif subject == 'S03':
    gemv_flatmaps_dict = gemv_flatmaps_dict_S03
df_gct = pd.DataFrame(gemv_flatmaps_dict).T

In [ ]:
def get_eng1000_weight_for_subject(subject):
    data = joblib.load(join(config.RESULTS_DIR_LOCAL, 'results_best_ensemble.pkl'))
    rr, cols_varied, mets = data['r'], data['cols_varied'], data['mets']
    metric_sort = 'corrs_tune_pc_weighted_mean'
    
    r = rr[
        (rr.feature_space == 'eng1000') * \
        (rr.num_stories == -1) * \
        (rr.feature_selection_alpha == -1) * \
        (rr.ndelays == 4)
    ]

    args = r[r.subject == 'S02'].iloc[0]
    model_params = joblib.load(
        join(args.save_dir_unique, 'model_params.pkl'))
    print(args.feature_space, args.pc_components, args.ndelays)
    wt = model_params['weights'] # wt is (n_delays x n_features) x n_voxels
    n_features = wt.shape[0] / args.ndelays
    wt = wt.reshape(args.ndelays, int(n_features), -1)
    wt = wt.mean(axis=0) # average over delays

    return wt, args['corrs_test']

wt, corrs_test = get_eng1000_weight_for_subject(subject)
# get top-10 percentile mask of corrs_test
mask_top = corrs_test >= np.percentile(corrs_test, 90)

In [ ]:
ENG1000_WORDS = neuro.analyze_helper.ENG1000_WORDS
# got these mappings by prompting GPT to """Return a dictionary where that maps each query to the closest semantic match in the database. Make sure every key is exactly in the queries list and every value is exactly in the database. Think hard."""
mappings = {
  'birthdays': 'birthday',
  'communication': 'talk',
  'death': 'die',
  'emotion': 'feel',
  'emotional expression': 'feel',
  'food preparation': 'cook',
  'hair and clothing': 'clothes',
  'laughter': 'laugh',
  'locations': 'place',
  'measurements': 'measure',
  'moments': 'moment',
  'negativity': 'bad',
  'physical injury or trauma': 'hurt',
  'rejection': 'no',
  'surprise': 'surprise',
  'time': 'hour',
  'abstract descriptions': 'idea',
  'cultural references': 'art',
  'dialogue': 'talk',
  'industry or profession': 'business',
  'negations': 'not',
  'numbers': 'number',
  'opinions or judgments': 'think',
  'personal or interactions interactions': 'person',
  'personal reflections or thoughts': 'think',
  'personal values or beliefs': 'trust',
  'physical actions': 'act',
  'planning or organizing': 'order',
  'proper nouns': 'name',
  'relationships between people': 'family',
  'sensory experiences': 'taste',
  'specific objects or items': 'object',
  'technical or specialized terminology': 'computer',
  'Body parts': 'body',
  'Descriptive elements of scenes or objects': 'picture',
  'Direction and location descriptions': 'direction',
  'Location names': 'city',
  'Personal growth and reflection': 'develop',
  'Scenes and settings': 'place',
  'Spatial positioning and directions': 'position',
  'Time and numbers': 'number',
  'Travel and location names': 'travel',
  'Unappetizing foods': 'poison',
  'Verbal interactions': 'speak',
  'Clothing and Physical Appearance': 'clothes',
  'Colors': 'colour',
  'Dialogue': 'talk',
  'Fear and Avoidance': 'fear',
  'Gruesome body imagery': 'blood',
  'Introspection': 'mind',
  'Measurements': 'measure',
  'Negative Emotional Reactions': 'sad',
  'Numbers': 'number',
  'Positive Emotional Reactions': 'happy',
  'Professions and Personal Backgrounds': 'job',
  'Recognition': 'notice',
  'Relationships': 'family',
  'Secretive Or Covert Actions': 'hide',
  'Sexual and Romantic Interactions': 'sex',
  'Times': 'hour',
  'Years': 'year'
}
ks = [k[0] for k in df_gct.index.tolist()]
for k, v in mappings.items():
    assert k in ks
    assert v in ENG1000_WORDS, v

for k in ks:
    assert k in mappings.keys()

In [ ]:
corrs = defaultdict(list)
for i in range(len(df_gct)):
    row = df_gct.iloc[i]
    word = df_gct.index[i][0]
    eng1000_word = mappings[word]
    eng1000_idx = ENG1000_WORDS.index(eng1000_word)
    weight_vector = wt[eng1000_idx, :]  # shape: n_voxels
    gct_vector = row.values  # shape: n_voxels

    # apply mask
    weight_vector = weight_vector[mask_top]
    gct_vector = gct_vector[mask_top]

    corr = np.corrcoef(gct_vector, weight_vector)[0,1]
    corrs['corr'].append(corr)
    corrs['word'].append(word)
    corrs['eng1000_word'].append(eng1000_word)
    corrs['expt'].append(df_gct.index[i][1])
    # print(f'Word: {word}, ENG1000 word: {eng1000_word}, Corr: {corr:.4f}')
corrs = pd.DataFrame(corrs)

# filter out non-GCT expts
corrs = corrs[(corrs.expt.notna()) & ~(corrs.expt == 'qa')]

In [ ]:
# visualize whole df
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(corrs.sort_values(by='corr', ascending=False))

In [ ]:
# remove bad matches
corrs = corrs[~corrs.word.isin([
    'rejection', 'emotion', 'emotional expression', 'time', 'communication'
])]

In [ ]:
sns.stripplot(corrs['corr'])

In [ ]:
# compute all pairwise correlations
dists = pairwise_distances(
    wt[:, mask_top],
    # wt[:, :],
    metric='correlation',
    n_jobs=-1,
)
# take upper diag of dists
dists = dists[np.triu_indices(dists.shape[0], k=1)]
dists = 1 - dists  # convert to correlations
# plt.hist(dists)

In [158]:
n_samples = len(corrs['corr'])
avg_corr = np.mean(corrs['corr'])

# compute p-value from null distribution
means = []
for i in tqdm(range(1000)):
    sampled_idxs = np.random.choice(len(dists), n_samples, replace=True)
    means.append(np.mean(dists[sampled_idxs]))
p_value = np.mean(np.array(means) >= avg_corr)
print(f'Average correlation: {avg_corr:.4f}, p-value: {p_value:.4f}')

100%|██████████| 1000/1000 [00:00<00:00, 29043.81it/s]

Average correlation: 0.2492, p-value: 0.0140


In [164]:
corrs[['word', 'eng1000_word', 'corr']].rename(
    columns={'word': 'GCT explanation', 'eng1000_word': 'Eng1000 word', 'corr': 'Correlation'}
).round(3).sort_values(by='Correlation', ascending=False)

,GCT explanation,Eng1000 word,Correlation
10,measurements,measure,0.457
5,food preparation,cook,0.428
7,laughter,laugh,0.427
0,birthdays,birthday,0.322
8,locations,place,0.268
9,locations,place,0.258
13,physical injury or trauma,hurt,0.242
12,negativity,bad,0.233
6,hair and clothing,clothes,0.182
11,moments,moment,0.166
